In [1]:
import numpy as np
import random
from collections import Counter
text=open("/content/smalltext8.txt").read()
text=text.split()

In [2]:
#Creating Vocabalary from given text
vocab=list(set(text))
#Creating dictionary for indexing in vocabalry
word2idx={}
idx2word={}
for i,w in enumerate(vocab):
  word2idx[w]=i
  idx2word[i]=w
V=len(vocab)

In [3]:
#Defining hyperparameters
dim=100
window=1
neg_samples=6
lr=0.025
epochs=1

#Finding probability of word in text
#This probability will be used to sample words from dictionary
freq=Counter(text)
p=np.array([freq[w] for w in vocab])
p=p**0.75
p=p/np.sum(p)

print(len(p))
print(V)


W=np.random.randn(V,dim)#Target Word Embeddings
C=np.random.randn(V,dim)#Context Word Embeddings

467
467


In [7]:
def sigmoid(z): return(1/(1+np.exp(-z)))
for _ in range(epochs):
    for i,w in enumerate(text):
        w=word2idx[w]#Target word
        context_ids=[word2idx[text[i+j]]#Positive Samples
                     for j in range(-window,window+1)
                     if j!=0 and i+j>=0 and i+j<len(text)]
        forbidden=set(context_ids+[w])#Samples which cannot be part of Negative Samples set

        #Use vectors and matrices to speed up computation
        for j in range(-window,window+1):
            if j==0 or i+j>=len(text) or i+j<0:continue
            #positive samples
            c=word2idx[text[i+j]]
            score=sigmoid(np.dot(W[w],C[c]))
            scale=lr*(score-1)
            W[w]=W[w]-scale*C[c]
            C[c]=C[c]-scale*W[w]

            #Negative samples
            neg=[]
            while(len(neg)<neg_samples):
                n=np.random.choice(V,p=p)
                #gives indices for negative samples
                if n not in forbidden:
                    neg.append(n)
            for n in neg:
                score=sigmoid(np.dot(W[w],C[c]))
                scale=lr*score
                W[w]=W[w]-scale*C[c]
                C[c]=C[c]-scale*W[w]
Emb=W#Final Word Embeddings
print(np.shape(Emb))

(467, 100)


In [8]:
def similar(word,top=5):
    v=Emb[word2idx[word]]
    #Comparing Embedding with all words in vocabalary
    sims=Emb@v/(np.linalg.norm(Emb,axis=1)*np.linalg.norm(V))
    #Evaluating words which have highest dot product
    ids=np.argsort(-sims)[1:top+1]
    return([idx2word[i] for i in ids])

print(similar("king"))

['where', 'french', 'l', 'simply', 'supported']
